In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,SimpleRNN, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score
from tensorflow.keras.utils import to_categorical
from sklearn.ensemble import RandomForestClassifier
import random
from sklearn.preprocessing import StandardScaler
import json
import joblib 

data = pd.read_csv('dataset/KU-HAR/3.Time_domain_subsamples/KU-HAR_time_domain_subsamples_20750x300.csv', header=None)
#AŞAĞIDAKİ YORUM SATIRINI AÇIP ÜSTTEKİNİ YORUMA ALIRSANIZ YENİDEN TRİN TEST ETMEK GEREKİR. ÜSTTEKİ BÜTÜN VERİ SETİ. ALTTAKI SUBSET
#data = pd.read_csv('dataset/KU-HAR/3.Time_domain_subsamples/KU-HAR_time_domain_subsamples_subset.csv', header=None)
activity_labels = [
    'Stand',
    'Sit',
    'Talk-sit',
    'Talk-stand',
    'Stand-sit',
    'Lay',
    'Lay-stand',
    'Pick',
    'Jump',
    'Push-up',
    'Sit-up',
    'Walk',
    'Walk-backward',
    'Walk-circle',
    'Run',
    'Stair-up',
    'Stair-down',
    'Table-tennis'
]
#print(data.head())
sensor_data = data.iloc[:, :1800].to_numpy()
labels = data.iloc[:,1800].to_numpy()
#(sensor_data.shape[0])

feature extraction for further use

In [2]:
timestemp_count = 300
feature_count = 6 # (acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z)
row_devided = sensor_data.reshape(sensor_data.shape[0], feature_count, timestemp_count)
X_3d = np.transpose(row_devided, (0, 2, 1))

print(X_3d.shape)

sensor_index = [
    (0, 300),    #acc_x
    (300, 600),   #acc_y
    (600, 900),   #acc_z
    (900, 1200),  #gyro_x
    (1200, 1500), #gyro_y
    (1500, 1800)  #gyro_z
]

feature_list = []

for i in range(sensor_data.shape[0]):
    segment_row = sensor_data[i]
    segment_features = []
    for j in range(len(sensor_index)):
        start, end = sensor_index[j]
        data = segment_row[start:end]

        segment_features.append(np.mean(data))
        segment_features.append(np.min(data))
        segment_features.append(np.max(data))
        segment_features.append(np.std(data))
        segment_features.append(np.median(data))
        segment_features.append(stats.median_abs_deviation(data)) # mad
        segment_features.append(np.sqrt(np.mean(data**2))) # rms
        segment_features.append(np.sum(data**2)) # enerji
        
        hist, _ = np.histogram(data, bins=10, density=True)
        segment_features.append(stats.entropy(hist))  # entropi

    feature_list.append(segment_features)

feature_extraction = pd.DataFrame(feature_list)
feature_names = ['mean', 'min', 'max', 'std', 'median', 'mad', 'rms', 'energy', 'entropy']
axis_name = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']

column_names = [f'{axis}_{feat}' for axis in axis_name for feat in feature_names]
feature_extraction.columns = column_names
labels = labels.ravel()
num_classes = len(np.unique(labels))
# print(feature_extraction.shape)
# print(feature_extraction.head())

(20750, 300, 6)


feature selection

In [3]:
rf_model = RandomForestClassifier(random_state=17)
rf_model.fit(feature_extraction, labels) #bütün veri ile eğitim

feature_importances = pd.DataFrame({
    'feature': column_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

feature_selection_num = 24
top_features = feature_importances.head(feature_selection_num)
selected_feature_names = top_features['feature'].tolist()

print(top_features)

X_features_selected = feature_extraction[selected_feature_names]

scaler = StandardScaler()
X_features_selected_scaled = scaler.fit_transform(X_features_selected)

features_3d = np.expand_dims(X_features_selected_scaled, axis=1)
features_corrert_size = np.tile(features_3d, (1, timestemp_count, 1))
X_combined = np.concatenate([X_3d, features_corrert_size], axis=2)
#print(X_combined.shape)
#print(X_combined)

          feature  importance
34  gyro_x_energy    0.037513
39     gyro_y_std    0.037503
33     gyro_x_rms    0.035469
43  gyro_y_energy    0.035206
14      acc_y_mad    0.033673
3       acc_x_std    0.033387
6       acc_x_rms    0.033285
30     gyro_x_std    0.032823
7    acc_x_energy    0.032686
23      acc_z_mad    0.032657
42     gyro_y_rms    0.032175
16   acc_y_energy    0.030317
5       acc_x_mad    0.030090
12      acc_y_std    0.029859
25   acc_z_energy    0.028800
41     gyro_y_mad    0.028380
32     gyro_x_mad    0.027949
24      acc_z_rms    0.025920
15      acc_y_rms    0.025750
21      acc_z_std    0.024397
48     gyro_z_std    0.019355
52  gyro_z_energy    0.019189
37     gyro_y_min    0.018420
2       acc_x_max    0.017949


simple rnn preparation



In [4]:
y_one_hot = to_categorical(labels, num_classes=num_classes)

print(y_one_hot.shape)
print(y_one_hot[0])

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_3d, y_one_hot, test_size=0.2, random_state=17, stratify=y_one_hot
) 
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=17, stratify=y_train_val
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(20750, 18)
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
(12450, 300, 6)
(4150, 300, 6)
(4150, 300, 6)


random search - simple rnn

bu kısım google colab ile çalıştırılıp sonuçlar kaydedilmiştir


In [ ]:
hp_optimization = {
    'units': [30, 40, 60, 80],            
    'dropout_rate': [0.3, 0.5, 0.7],
    'batch_size': [32, 64]             
}
num_tries = 20  

results_history = []
best_f1_score = 0.0
best_params = {}
best_model = None

for i in range(num_tries):

    current_params = {
        'units': random.choice(hp_optimization['units']),
        'dropout_rate': random.choice(hp_optimization['dropout_rate']),
        'batch_size': random.choice(hp_optimization['batch_size'])
    }
    
    print(f"\n Try {i+1}/{num_tries} | Parameters: {current_params}")
   
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
    model.add(SimpleRNN(units=current_params['units']))
    model.add(Dropout(current_params['dropout_rate']))
    model.add(Dense(units=num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    history = model.fit(
        X_train, 
        y_train, 
        epochs=50,  
        batch_size=current_params['batch_size'], 
        validation_data=(X_val, y_val),
        verbose=0 
    )
    
    y_val_pred_probs = model.predict(X_val)
    y_val_pred = np.argmax(y_val_pred_probs, axis=1)
    y_val_true = np.argmax(y_val, axis=1)
    
    val_f1_score = f1_score(y_val_true, y_val_pred, average='macro')
    val_accuracy = history.history['val_accuracy'][-1] 
    print(f"Validation Macro F1-Score: {val_f1_score:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

    result_entry = current_params.copy()
    result_entry['f1_score'] = val_f1_score
    result_entry['accuracy'] = history.history['val_accuracy'][-1]
    results_history.append(result_entry)

    if val_f1_score > best_f1_score:
        print(f"BEST MODEL (F1-Score: {val_f1_score:.4f}, Accuracy: {val_accuracy:.4f})")
        best_f1_score = val_f1_score
        best_params = current_params
        best_model = model

print(f"Best Validation F1: {best_f1_score:.4f}")
print(f"Bst parameters: {best_params}")


 Try 1/20 | Parameters: {'units': 80, 'dropout_rate': 0.7, 'batch_size': 32}


save simple rnn model

bu kısım google colab ile çalıştırılıp sonuçlar kaydedilmiştir


In [ ]:
results_df = pd.DataFrame(results_history)
results_df_sorted = results_df.sort_values(by='f1_score', ascending=False)
results_df_sorted.to_csv('models_for_yontem2/kuhar/optimization_results_kuhar_yontem2.csv', index=False)
print(">>> Tüm optimizasyon denemeleri 'optimization_results_kuhar_yontem2.csv' dosyasına kaydedildi.")

with open('models_for_yontem2/kuhar/best_params_kuhar_yontem2.json', 'w') as f:
    json.dump(best_params, f, indent=4)

best_model.save('models_for_yontem2/kuhar/best_simple_rnn_model_kuhar_yontem2.keras')

rnn with feature selection

In [5]:
X_train_val_with_fs, X_test_with_fs, y_train_val_with_fs, y_test_with_fs = train_test_split(
    X_combined, y_one_hot, test_size=0.2, random_state=17, stratify=y_one_hot
) 
X_train_with_fs, X_val_with_fs, y_train_with_fs, y_val_with_fs = train_test_split(
    X_train_val_with_fs, y_train_val_with_fs, test_size=0.25, random_state=17, stratify=y_train_val_with_fs
)

with open('models_for_yontem2/kuhar/best_params_kuhar_yontem2.json', 'r') as f:
        best_params = json.load(f)


model_with_fs = Sequential()
model_with_fs.add(Input(shape=(X_combined.shape[1], X_combined.shape[2])))
model_with_fs.add(SimpleRNN(units=best_params['units']))
model_with_fs.add(Dropout(best_params['dropout_rate']))
model_with_fs.add(Dense(units=num_classes, activation='softmax'))

model_with_fs.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model_with_fs.fit(
    X_train_with_fs,
    y_train_with_fs, 
    epochs=50,
    batch_size=best_params['batch_size'], 
    validation_data=(X_val_with_fs, y_val_with_fs),
    verbose=1 
)
model_with_fs.save('models_for_yontem2/kuhar/simple_rnn_with_fs_kuhar_yontem2.keras')
joblib.dump(scaler, 'models_for_yontem2/kuhar/simple_rnn_with_fs_scaler.joblib')
with open('models_for_yontem2/kuhar/selected_feature_names.txt', 'w') as f:
    for feature_name in selected_feature_names:
        f.write(f"{feature_name}\n")


Epoch 1/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 23s 51ms/step - accuracy: 0.1158 - loss: 2.7992 - val_accuracy: 0.1316 - val_loss: 2.4699
Epoch 2/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 17s 43ms/step - accuracy: 0.1385 - loss: 2.4962 - val_accuracy: 0.1677 - val_loss: 2.2776
Epoch 3/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 16s 42ms/step - accuracy: 0.1396 - loss: 2.4795 - val_accuracy: 0.1494 - val_loss: 2.3749
Epoch 4/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 15s 39ms/step - accuracy: 0.1566 - loss: 2.3959 - val_accuracy: 0.1749 - val_loss: 2.2765
Epoch 5/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 19s 48ms/step - accuracy: 0.1719 - loss: 2.3019 - val_accuracy: 0.1641 - val_loss: 2.3043
Epoch 6/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 18s 46ms/step - accuracy: 0.1783 - loss: 2.2692 - val_accuracy: 0.2108 - val_loss: 2.1041
Epoch 7/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.1902 - loss: 2.2220 - val_accuracy: 0.1855 - val_loss: 2.1947
Epoch 8/50
390/390 ━━━━━━━━━━━━━━━━━━━━ 32s 28ms/step - accuracy: 0.1884 - loss: 2.2097 - 